In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import seaborn as sns
import os
from collections import Counter
from sklearn.utils import class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support

# --- 1. CONSTANTS AND PATHS ---
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS_PHASE_1 = 15  # Fast training for the new layers
EPOCHS_PHASE_2 = 50  # Slow fine-tuning for the whole model

train_dir = "C:/Users/awadh/Wheat/data/train"
val_dir = "C:/Users/awadh/Wheat/data/valid"
test_dir = "C:/Users/awadh/Wheat/data/test"

# --- 2. DATA GENERATORS (Using the robust ImageDataGenerator) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical')

val_generator = val_datagen.flow_from_directory(
    val_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
    test_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=1, class_mode='categorical', shuffle=False)

CLASS_NAMES = list(train_generator.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print("Detected Classes:", CLASS_NAMES)

# --- 3. CLASS WEIGHTING (TO FIX IMBALANCE) ---
class_counts = Counter(train_generator.classes)
print("Class Counts:", class_counts)

class_labels = train_generator.classes
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(class_labels),
    y=class_labels
)
class_weight_dict = dict(zip(np.unique(class_labels), class_weights))
print("Class Weights Dictionary:", class_weight_dict)

# --- 4. BUILD THE MODEL ---
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# Build the "head"
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)
classification_model = Model(inputs=base_model.input, outputs=predictions)

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
    ModelCheckpoint("best_model.keras", monitor='val_loss', save_best_only=True)
]

# --- 5. PHASE 1: FEATURE EXTRACTION (FAST) ---
print("\n--- STARTING PHASE 1: FEATURE EXTRACTION ---")
# Freeze the base model layers
base_model.trainable = False

# Compile the model with a HIGHER learning rate
classification_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                             loss='categorical_crossentropy',
                             metrics=['accuracy'])

# Train *only* the new head
history_phase_1 = classification_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_PHASE_1,
    callbacks=callbacks,
    class_weight=class_weight_dict
)
print("--- PHASE 1 COMPLETE ---")

# --- 6. PHASE 2: FINE-TUNING (SLOW) ---
print("\n--- STARTING PHASE 2: FINE-TUNING ---")
# Unfreeze the whole model
base_model.trainable = True

# Re-compile the model with a VERY LOW learning rate
classification_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # 100x smaller
                             loss='categorical_crossentropy',
                             metrics=['accuracy'])

# Continue training the whole model
history_phase_2 = classification_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_PHASE_2,
    callbacks=callbacks,
    class_weight=class_weight_dict
)
print("--- PHASE 2 COMPLETE ---")

# Save the final, fine-tuned model
classification_model.save("wheat_disease_classifier_final.keras")

# --- 7. PLOT TRAINING HISTORY ---
def plot_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title("Fine-Tuning: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title("Fine-Tuning: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_history(history_phase_2)

# --- 8. PREDICTION FUNCTION ---
def predict_image(image_path):
    img = load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)
    pred = classification_model.predict(img_batch)
    class_idx = np.argmax(pred)
    confidence = pred[0][class_idx]
    
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Class: {CLASS_NAMES[class_idx]}, Confidence: {confidence:.2f}")
    plt.show()

print("\n--- EXAMPLE PREDICTION ---")
predict_image("/kaggle/input/wheat-plant-diseases/data/train/Black Rust/black_rust_1.png")

# --- 9. FULL EVALUATION ON TEST DATA ---
print("\n--- EVALUATING ON TEST DATA ---")
test_generator.reset()
preds = classification_model.predict(test_generator, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()
# Classification Report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Precision, Recall, F1-Score Bar Graph
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(NUM_CLASSES))
metrics_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1})

print("\nDetailed Metrics per Class:")
print(metrics_df)

metrics_df.set_index('Class')[['Precision', 'Recall', 'F1-score']].plot(kind='bar', figsize=(12, 6), ylim=(0, 1))
plt.title("Precision, Recall, and F1-Score per Class")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

Found 13104 images belonging to 15 classes.
Found 300 images belonging to 15 classes.
Found 750 images belonging to 15 classes.
Detected Classes: ['Aphid', 'Black Rust', 'Blast', 'Brown Rust', 'Common Root Rot', 'Fusarium Head Blight', 'Healthy', 'Leaf Blight', 'Mildew', 'Mite', 'Septoria', 'Smut', 'Stem fly', 'Tan spot', 'Yellow Rust']
Class Counts: Counter({11: 1310, 14: 1301, 3: 1271, 10: 1144, 8: 1081, 6: 1000, 0: 903, 7: 842, 9: 800, 13: 770, 2: 647, 4: 614, 5: 611, 1: 576, 12: 234})
Class Weights Dictionary: {0: 0.9674418604651163, 1: 1.5166666666666666, 2: 1.3502318392581143, 3: 0.6873328088119591, 4: 1.422801302931596, 5: 1.4297872340425533, 6: 0.8736, 7: 1.0375296912114014, 8: 0.8081406105457909, 9: 1.092, 10: 0.7636363636363637, 11: 0.6668702290076336, 12: 3.7333333333333334, 13: 1.1345454545454545, 14: 0.6714834742505765}

--- STARTING PHASE 1: FEATURE EXTRACTION ---
Epoch 1/15
410/410 [==============================] - ETA: 0s - loss: 2.9919 - accuracy: 0.0882

TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.